# Hand classifier
This model identifies the number of extended fingers based on a clear image of a human hand.

In [1]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
from keras import layers

import numpy as np
import matplotlib.pyplot as plt

from keras import backend as K
print(K.backend())

2026-04-10 14:38:25.202695: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


tensorflow


In [2]:
from sklearn.model_selection import train_test_split
batch_size_n = 4
img_size = (224, 224)

train_dataset, validation_dataset = keras.utils.image_dataset_from_directory(
    'images/train',
    batch_size=batch_size_n, 
    image_size=img_size,
    crop_to_aspect_ratio=True,
    label_mode='categorical',
    validation_split=0.177, # ensures validation and test sets are equal size
    subset='both',
    shuffle=True,
    seed=123)

test_dataset = keras.utils.image_dataset_from_directory(
    'images/test',
    batch_size=batch_size_n, 
    image_size=img_size,
    crop_to_aspect_ratio=True,
    shuffle=False,
    label_mode='categorical')

Found 255 files belonging to 5 classes.
Using 210 files for training.
Using 45 files for validation.


I0000 00:00:1775821108.392787   60315 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1767 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Found 45 files belonging to 5 classes.


In [3]:
from keras import Sequential
from keras import layers
from keras.optimizers import Adam
num_classes = 5

model = Sequential([
    layers.Input(shape=(224, 224, 3)),
    
    # Processing / augmentation
    layers.Rescaling(1./255),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),

    # Convolution
    layers.Conv2D(32, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Conv2D(128, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),

    # Output
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(optimizer=Adam(learning_rate = 0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
history = model.fit(train_dataset, batch_size=4, epochs=50, validation_data=validation_dataset)

Epoch 1/50


2026-04-10 14:38:31.844121: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 92000


53/53 ━━━━━━━━━━━━━━━━━━━━ 5s 39ms/step - accuracy: 0.1810 - loss: 1.6288 - val_accuracy: 0.1556 - val_loss: 1.6095
Epoch 2/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.2238 - loss: 1.6091 - val_accuracy: 0.3333 - val_loss: 1.5978
Epoch 3/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.3238 - loss: 1.5844 - val_accuracy: 0.4000 - val_loss: 1.5498
Epoch 4/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.3095 - loss: 1.5471 - val_accuracy: 0.3111 - val_loss: 1.4916
Epoch 5/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s -862114us/step - accuracy: 0.3381 - loss: 1.4827 - val_accuracy: 0.4222 - val_loss: 1.3946
Epoch 6/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.3857 - loss: 1.4346 - val_accuracy: 0.2889 - val_loss: 1.3801
Epoch 7/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.3667 - loss: 1.4397 - val_accuracy: 0.4889 - val_loss: 1.2550
Epoch 8/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.3905 - loss: 1.3590 - val_accuracy: 0.4222 - val_los

In [ ]:
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs = range(1, len(acc) + 1)
plt.plot(epochs, acc, "r--", label="Training accuracy")
plt.plot(epochs, val_acc, "b", label="Validation accuracy")
plt.title("Training and validation accuracy")
plt.legend()
plt.figure()
plt.plot(epochs, loss, "r--", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.legend()
plt.show()

In [ ]:
loss, acc = model.evaluate(test_dataset, verbose=0)
print(f"The model's test accuracy is {acc}")

The convolutional model that was trained with these images from the beginning achieved an accuracy between 66-68%.
With 5 classes this means that the model has found some patterns and isn't just making wild guesses.  

Next, visualizing the model's predictions might make it easier to understand.

In [ ]:
y_pred = model.predict(test_dataset)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(np.concatenate([y for x, y in test_dataset], axis=0), axis=1)

print(y_pred_classes)
print(y_true_classes)

In [ ]:
images = []
labels = []
class_names = ['1 finger', '2 fingers', '3 fingers', '4 fingers', '5 fingers']

for image_batch, label_batch in test_dataset:
    images.append(image_batch.numpy())
    labels.append(label_batch.numpy())

images = np.concatenate(images)
labels = np.concatenate(labels)

In [ ]:
def show_example_images_and_predictions(predicted_labels):
    plt.figure(figsize=(15, 8))
    
    for i in range(20):
        plt.subplot(4, 5, i + 1)
    
        plt.imshow(images[i].astype("uint8"))
    
        true_label = y_true_classes[i]
        pred_label = predicted_labels[i]
    
        color = "green" if true_label == pred_label else "red"
    
        plt.title(
            f"True: {class_names[true_label]}\nPred: {class_names[pred_label]}",
            color=color
        )
    
        plt.axis("off")
    
    plt.tight_layout()
    plt.show()

In [ ]:
show_example_images_and_predictions(y_pred_classes)

## VGG16 Pretrained model

In [ ]:
import tensorflow as tf
from keras.applications import VGG16

conv_base = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

In [ ]:
from keras.applications.vgg16 import preprocess_input

def preprocess(images):
    return preprocess_input(tf.cast(images, tf.float32))

In [ ]:
# code from https://deeplearningwithpython.io/chapters/chapter08_image-classification/#using-a-pretrained-model
def get_features_and_labels(dataset):
    all_features = []
    all_labels = []
    for images, labels in dataset:
        preprocessed_images = preprocess(images)
        features = conv_base.predict(preprocessed_images, verbose=0)
        all_features.append(features)
        all_labels.append(labels)
    return np.concatenate(all_features), np.concatenate(all_labels)

train_features, train_labels = get_features_and_labels(train_dataset)
val_features, val_labels = get_features_and_labels(validation_dataset)
test_features, test_labels = get_features_and_labels(test_dataset)

In [ ]:
# check that shapes match as expected
print(train_features.shape)
print(train_labels.shape)

In [ ]:
model = Sequential([
    layers.Input(shape=(7, 7, 512)),
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="feature_extraction.keras",
        save_best_only=True,
        monitor="val_loss",
    )
]

history = model.fit(
    train_features,
    train_labels,
    epochs=20,
    validation_data=(val_features, val_labels),
    callbacks=callbacks,
)

In [ ]:
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs = range(1, len(acc) + 1)
plt.plot(epochs, acc, "r--", label="Training accuracy")
plt.plot(epochs, val_acc, "b", label="Validation accuracy")
plt.title("Training and validation accuracy")
plt.legend()
plt.figure()
plt.plot(epochs, loss, "r--", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.legend()
plt.show()

In [ ]:
loss, acc = model.evaluate(test_features, test_labels, verbose=0)
print(f"The model's test accuracy is {acc}")

In [ ]:
y_pred = model.predict(test_features)
y_pred_classes = np.argmax(y_pred, axis=1)
show_example_images_and_predictions(y_pred_classes)